# MAI 510 | Module 4 Instructor Colab
## Predicting Customer Churn: two assignments, one reproducible experiment

**Part 1 — Assignment 4.1:** dataset audit, one-hot inputs, dense network, five metrics, confusion matrix and the top-15% contact decision.

**Part 2 — Assignment 4.2:** entity embeddings, gradient-boosted baseline, consistent evaluation and a deployment recommendation.

**Final step:** generate two named Word report templates and **one ZIP** containing dataset, split IDs, models, charts, dashboards, tables and proof of work. The last cell offers the ZIP download.

**Teaching notes.** Run cells in order. Start with `DEMO_MODE=True` for a quicker lesson. Students should use `DEMO_MODE=False` for their final experiment. The notebook records measured values from executed code; it never inserts invented results. Assignment 4.2 is optional when demonstrating only the first assignment.

## LAB 00 — Student or instructor identity and run configuration
**Why?** Every output is created in a folder containing the student's name. Change `STUDENT_NAME` before running. You can use any first-hand or fictional company context; the required dataset remains the IBM Telco Customer Churn CSV.

In [ ]:
STUDENT_NAME = "Ivan Suarez"  # @param {type:"string"}
COURSE = "MAI 510 — Deep Learning"
MODULE = "Module 4"
COMPANY = "ConnectPlus Telecommunications (fictional case)"
DEMO_MODE = True  # @param {type:"boolean"}
RUN_PART_2 = True  # @param {type:"boolean"}
RANDOM_SEED = 42
CONTACT_FRACTION = 0.15

import re, os, json, time, math, random, zipfile, hashlib, sys
from pathlib import Path
from datetime import datetime, timezone
assert STUDENT_NAME.strip(), "Enter your full name in STUDENT_NAME."
SAFE_NAME = re.sub(r'[^A-Za-z0-9]+','_',STUDENT_NAME.strip()).strip('_')
assert 0 < CONTACT_FRACTION < 1
np_seed = RANDOM_SEED
random.seed(RANDOM_SEED)
ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
OUT = ROOT / f'MAI510_M4_{SAFE_NAME}_{RUN_ID}'
ART = OUT/'artifacts'; ART.mkdir(parents=True,exist_ok=True)
CHARTS = ART/'charts'; CHARTS.mkdir(exist_ok=True)
MODELS = ART/'models'; MODELS.mkdir(exist_ok=True)
TABLES = ART/'tables'; TABLES.mkdir(exist_ok=True)
REPORTS = ART/'reports'; REPORTS.mkdir(exist_ok=True)
PROOF = ART/'proof'; PROOF.mkdir(exist_ok=True)
print('Student:',STUDENT_NAME, '| Mode:', 'QUICK INSTRUCTOR DEMO' if DEMO_MODE else 'FULL EXPERIMENT')
print('Output folder:',OUT)

## LAB 01 — Install and import tools

Core work uses **Pandas** (tables), **scikit-learn** (splits, transformations, metrics and boosting), **TensorFlow/Keras** (neural networks), **Plotly** (interactive dashboards), **Matplotlib/Seaborn** (saved figures), and **python-docx** (Word reports). Colab usually includes most libraries. Installation can take a minute.

In [ ]:
%pip -q install python-docx plotly ipywidgets
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,
                             confusion_matrix,ConfusionMatrixDisplay,roc_curve,precision_recall_curve)
from docx import Document
from docx.shared import Inches
import tensorflow as tf
tf.keras.utils.set_random_seed(RANDOM_SEED)
print('Python:',sys.version.split()[0],'TensorFlow:',tf.__version__,'scikit-learn available')

## LAB 02 — Upload the supplied IBM dataset

Use the exact CSV from the Module 4 resource package: **`IBM_Telco_Customer_Churn.csv`**. Click **Choose Files** when prompted. The notebook checks that this has the expected 7,043 rows and that `Churn` exists. Do not substitute a synthetic data file without making that change explicit in your report.

In [ ]:
EXPECTED_FILE = 'IBM_Telco_Customer_Churn.csv'
try:
    from google.colab import files
    print('Upload the provided IBM_Telco_Customer_Churn.csv file now.')
    uploaded = files.upload()
    chosen = Path(EXPECTED_FILE) if Path(EXPECTED_FILE).exists() else Path(next(iter(uploaded)))
except ImportError:
    chosen = Path(EXPECTED_FILE)
    assert chosen.exists(), f'Place {EXPECTED_FILE} beside this notebook.'
df_raw = pd.read_csv(chosen, dtype={'TotalCharges':'string'})
assert df_raw.shape == (7043,21), f'Expected IBM standard CSV (7043,21), received {df_raw.shape}'
assert {'customerID','Churn','TotalCharges','Contract'}.issubset(df_raw.columns)
assert df_raw.customerID.is_unique and set(df_raw.Churn.dropna().unique())=={'Yes','No'}
DATA_PATH = ART/EXPECTED_FILE
df_raw.to_csv(DATA_PATH,index=False)
DATA_SHA256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print('Loaded',df_raw.shape,'SHA256:',DATA_SHA256)
display(df_raw.head(5))

# PART 1 — ASSIGNMENT 4.1
## LAB 03 — What is in the data?

An **observation** is one customer. A **feature** is a predictor such as tenure. The **target** (`Churn`) is what the classifier tries to predict. Count class frequencies and inspect data types; a high accuracy score can hide poor detection of the less frequent class.

In [ ]:
churn_counts = df_raw.Churn.value_counts().rename_axis('Churn').reset_index(name='Customers')
display(churn_counts)
audit = pd.DataFrame({'dtype':df_raw.dtypes.astype(str), 'pandas_missing':df_raw.isna().sum(),
                      'unique_values':df_raw.nunique(dropna=False)})
audit.to_csv(TABLES/'A41_original_data_audit.csv')
display(audit)
fig=px.bar(churn_counts,x='Churn',y='Customers',color='Churn',title='IBM sample: observed customer churn',text='Customers',
           color_discrete_map={'Yes':'#e76f51','No':'#117c83'})
fig.write_html(str(CHARTS/'A41_interactive_churn_distribution.html'))
display(fig)

## LAB 04 — Investigate missing values and save a cleaning record

`TotalCharges` looks numeric but the IBM CSV stores it as text. Eleven rows contain blank values. These blanks are **missing**, whereas “No internet service” is a legitimate categorical response. For reproducibility, this notebook retains all rows and imputes missing numeric values using the training median *after* the split.

In [ ]:
df=df_raw.copy()
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
invalid_total=int(df.TotalCharges.isna().sum())
print('Unparseable TotalCharges:',invalid_total)
display(df.loc[df.TotalCharges.isna(),['customerID','tenure','MonthlyCharges','TotalCharges']].head(15))
assert invalid_total==11, 'Dataset missing-value pattern changed: review cleaning decisions.'
cleaning_log=[
 {'step':'Parse TotalCharges','decision':'Convert text to numeric with errors=coerce','reason':f'{invalid_total} original blanks cannot be parsed'},
 {'step':'Missing numeric values','decision':'Impute median learned from training partition only','reason':'Retain customers without leaking holdout statistics'},
 {'step':'customerID','decision':'Exclude from predictors; retain for split and proof','reason':'Identifier is not a generalizable customer characteristic'},
 {'step':'Churn','decision':'Map Yes to 1 and No to 0','reason':'Binary classification target'},
 {'step':'No internet service category','decision':'Preserve categorical response','reason':'It is a real value rather than missing data'},
]
pd.DataFrame(cleaning_log).to_csv(TABLES/'A41_cleaning_decisions.csv',index=False)

## LAB 05 — Create the fixed train, validation and test populations

**Training** teaches weights. **Validation** selects settings and the 15% decision policy. **Test** is untouched until final evaluation. Using the same IDs again in Assignment 4.2 ensures a fair comparison. Stratification keeps churn representation similar in each partition.

In [ ]:
df['ChurnFlag']=(df['Churn']=='Yes').astype(int)
rows=np.arange(len(df)); y_all=df.ChurnFlag.to_numpy()
train_idx,holdout_idx=train_test_split(rows,test_size=.40,random_state=RANDOM_SEED,stratify=y_all)
val_idx,test_idx=train_test_split(holdout_idx,test_size=.50,random_state=RANDOM_SEED,stratify=y_all[holdout_idx])
assert len(set(train_idx)&set(val_idx))==0 and len(set(train_idx)&set(test_idx))==0
split_df=pd.DataFrame({'customerID':df.customerID,'split':'unassigned'})
split_df.loc[train_idx,'split']='train';split_df.loc[val_idx,'split']='validation';split_df.loc[test_idx,'split']='test'
assert (split_df['split']=='unassigned').sum()==0
split_df.to_csv(TABLES/'REQUIRED_reproducible_customer_splits.csv',index=False)
df_train=df.iloc[train_idx].copy();df_val=df.iloc[val_idx].copy();df_test=df.iloc[test_idx].copy()
y_train=df_train.ChurnFlag.to_numpy(); y_val=df_val.ChurnFlag.to_numpy(); y_test=df_test.ChurnFlag.to_numpy()
print('Train / validation / test:',len(train_idx),len(val_idx),len(test_idx))
display(split_df['split'].value_counts())
pd.DataFrame({'split':['train','validation','test'], 'n':[len(train_idx),len(val_idx),len(test_idx)],
              'churn_rate':[y_train.mean(),y_val.mean(),y_test.mean()]}).to_csv(TABLES/'A41_split_summary.csv',index=False)

## LAB 06 — Numerical scaling and categorical one-hot encoding

**One-hot encoding** creates a separate 0/1 column for each category. **Standard scaling** expresses numerical features relative to the training mean and standard deviation. Both transformations must learn their parameters from training data only. This lab uses the same feature information again for gradient boosting in Part 2.

In [ ]:
NUMERIC=['tenure','MonthlyCharges','TotalCharges']
CATEGORICAL=[c for c in df.columns if c not in NUMERIC+['customerID','Churn','ChurnFlag']]
FEATURES=NUMERIC+CATEGORICAL
num_pipe=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
try:
    encoder=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
except TypeError:
    encoder=OneHotEncoder(handle_unknown='ignore',sparse=False)
prep=ColumnTransformer([('numerical',num_pipe,NUMERIC),('categorical',encoder,CATEGORICAL)],sparse_threshold=0)
X_train=prep.fit_transform(df_train[FEATURES]).astype('float32')
X_val=prep.transform(df_val[FEATURES]).astype('float32')
X_test=prep.transform(df_test[FEATURES]).astype('float32')
assert X_train.shape[1]==X_val.shape[1]==X_test.shape[1]
print('Feature groups:',len(NUMERIC),'numeric and',len(CATEGORICAL),'categorical')
print('One-hot matrix dimensions:',X_train.shape,'and test',X_test.shape)
try:
    import joblib
    joblib.dump(prep,MODELS/'A41_onehot_training_fitted_preprocessor.joblib')
except Exception as e: print('Preprocessor save failed:',e)

## LAB 07 — Train a dense neural network

A **dense layer** connects each input to learned weights; **ReLU** activates nonlinear patterns; **dropout** randomly disables part of a hidden representation during training; **sigmoid** outputs a score between zero and one. An **epoch** means one pass over training rows. Early stopping watches validation loss.

In [ ]:
tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(RANDOM_SEED)
EPOCHS = 8 if DEMO_MODE else 65
PATIENCE = 2 if DEMO_MODE else 7
BATCH_SIZE = 64
dense_net=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(32,activation='relu'),
    tf.keras.layers.Dense(1,activation='sigmoid'),
],name='A41_dense_churn_network')
dense_net.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='binary_crossentropy',metrics=['accuracy'])
dense_net.summary()
t0=time.perf_counter()
dense_history=dense_net.fit(X_train,y_train,validation_data=(X_val,y_val),
    epochs=EPOCHS,batch_size=BATCH_SIZE,verbose=1,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=PATIENCE,restore_best_weights=True)])
dense_seconds=time.perf_counter()-t0
dense_net.save(MODELS/'A41_dense_onehot.keras')
hist=pd.DataFrame(dense_history.history);hist.index=hist.index+1;hist.index.name='epoch'
hist.to_csv(TABLES/'A41_training_history.csv')
print('Fit seconds:',round(dense_seconds,2),'epochs completed:',len(hist),'best val loss epoch:',int(hist.val_loss.idxmin()))

## LAB 08 — Training curves and overfitting explanation

Training loss decreasing while validation loss starts increasing can indicate **overfitting**. The plot reflects the actual executed run. A short demo may stop before clear overfitting occurs; students should say so rather than invent a point.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,4.5))
ax[0].plot(hist.index,hist.loss,label='Training loss');ax[0].plot(hist.index,hist.val_loss,label='Validation loss')
ax[0].axvline(hist.val_loss.idxmin(),color='black',linestyle='--',label='Best validation loss')
ax[0].set_title('Dense model loss');ax[0].set_xlabel('Epoch');ax[0].legend()
ax[1].plot(hist.index,hist.accuracy,label='Training accuracy');ax[1].plot(hist.index,hist.val_accuracy,label='Validation accuracy')
ax[1].set_title('Dense model accuracy');ax[1].set_xlabel('Epoch');ax[1].legend()
fig.tight_layout();fig.savefig(CHARTS/'A41_training_validation_curves.png',dpi=160);plt.show()

## LAB 09 — Evaluate the five required metrics

**Accuracy:** correct share overall. **Precision:** flagged customers who truly churned. **Recall:** actual churners flagged. **F1:** balance of precision and recall. **ROC-AUC:** ability to rank churners above non-churners across cutoffs. Report 0.50-based classification metrics before the separate capacity decision.

In [ ]:
def scores_at_cutoff(y,prob,cutoff=.5):
    pred=(np.asarray(prob)>=cutoff).astype(int)
    return {'accuracy':accuracy_score(y,pred),'precision':precision_score(y,pred,zero_division=0),
            'recall':recall_score(y,pred,zero_division=0),'f1':f1_score(y,pred,zero_division=0),
            'roc_auc':roc_auc_score(y,prob)}

dense_prob=dense_net.predict(X_test,verbose=0).reshape(-1)
dense_val_prob=dense_net.predict(X_val,verbose=0).reshape(-1)
dense_metrics=scores_at_cutoff(y_test,dense_prob,.5)
pd.DataFrame([{'model':'Dense one-hot',**dense_metrics,'train_seconds':dense_seconds,
               'trainable_parameters':dense_net.count_params()}]).to_csv(TABLES/'A41_five_test_metrics.csv',index=False)
display(pd.DataFrame([dense_metrics]).style.format('{:.3f}'))

cm=confusion_matrix(y_test,dense_prob>=.5,labels=[0,1]);assert cm.sum()==len(test_idx)
fig,ax=plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(cm,display_labels=['Stayed','Churned']).plot(ax=ax,cmap='Blues',values_format='d',colorbar=False)
ax.set_title('Assignment 4.1 confusion matrix at 0.50');fig.tight_layout()
fig.savefig(CHARTS/'A41_confusion_matrix_default_050.png',dpi=160);plt.show()
pd.DataFrame(cm,index=['actual_stayed','actual_churned'],columns=['pred_stayed','pred_churned']).to_csv(TABLES/'A41_confusion_matrix_counts.csv')
print('TN, FP, FN, TP:',cm[0,0],cm[0,1],cm[1,0],cm[1,1])

## LAB 10 — Precision and recall as the contact budget changes

The company can contact only **15%** of customers. A probability score is first used to **rank** customers, then the retention team selects the highest scores. A capacity ranking is different from blindly using 0.50. This chart shows precision and recall at contact fractions from 5% to 50% on the validation partition.

In [ ]:
def rank_policy(y,prob,fraction=CONTACT_FRACTION):
    prob=np.asarray(prob).reshape(-1); k=max(1,int(math.ceil(fraction*len(prob))))
    order=np.argsort(-prob,kind='stable');flags=np.zeros(len(prob),dtype=int);flags[order[:k]]=1
    return flags,k,float(prob[order[k-1]])

policy_rows=[]
for frac in np.arange(.05,.51,.05):
    flags,k,boundary=rank_policy(y_val,dense_val_prob,float(frac))
    policy_rows.append({'contact_fraction':frac,'contacts':k,'precision':precision_score(y_val,flags,zero_division=0),
                        'recall':recall_score(y_val,flags,zero_division=0),'boundary_score':boundary})
policy_df=pd.DataFrame(policy_rows);policy_df.to_csv(TABLES/'A41_validation_capacity_sweep.csv',index=False)
chart=px.line(policy_df,x='contact_fraction',y=['precision','recall'],markers=True,
              title='Validation: model effectiveness versus available outreach capacity',labels={'value':'Score','contact_fraction':'Fraction contacted'})
chart.add_vline(x=CONTACT_FRACTION,line_dash='dash',line_color='black')
chart.write_html(str(CHARTS/'A41_interactive_validation_precision_recall_vs_capacity.html'))
display(chart)

## LAB 11 — Freeze the top-15% policy and report held-out results

To honor a strict contact budget, the reproducible policy is to sort each batch of customers by descending risk, then select `ceil(0.15 × N)`. The score of the last selected validation customer is shown as a **reference boundary**, not treated as a guaranteed 15% numeric cutoff for every new population. The test labels are used only for reporting performance, never to decide who gets selected.

In [ ]:
val_flags,val_k,val_boundary=rank_policy(y_val,dense_val_prob)
test_flags,test_k,test_boundary=rank_policy(y_test,dense_prob)
top15_precision=precision_score(y_test,test_flags,zero_division=0)
top15_recall=recall_score(y_test,test_flags,zero_division=0)
capacity_result={'contact_fraction':CONTACT_FRACTION,'validation_contacts':int(val_k),
                 'validation_reference_score':val_boundary,'test_customers':len(y_test),
                 'test_contacts':int(test_k),'test_precision':float(top15_precision),
                 'test_recall':float(top15_recall),
                 'policy':'Rank by descending predicted churn score; contact ceil(15% of evaluated population)'}
(PROOF/'REQUIRED_A41_contact_policy.json').write_text(json.dumps(capacity_result,indent=2))
display(pd.DataFrame([capacity_result]))
test_predictions=pd.DataFrame({'customerID':df_test.customerID,'actual_churn':y_test,
                               'dense_probability':dense_prob,'dense_predicted_at_050':(dense_prob>=.5).astype(int),
                               'selected_top_15pct':test_flags})
test_predictions.to_csv(TABLES/'A41_test_prediction_evidence.csv',index=False)
cm15=confusion_matrix(y_test,test_flags,labels=[0,1])
fig,ax=plt.subplots(figsize=(6,5));ConfusionMatrixDisplay(cm15,display_labels=['Stayed','Churned']).plot(ax=ax,cmap='Greens',colorbar=False)
ax.set_title('Test contact policy: top 15% ranked by risk');fig.tight_layout()
fig.savefig(CHARTS/'A41_top15_contact_confusion_matrix.png',dpi=160);plt.show()

## LAB 12 — Part 1 interactive dashboard

The Plotly dashboard is interactive inside Colab and saved as a standalone HTML file. It contains model test metrics, the class distribution and the observed contact policy. The dashboard does not change the underlying model or data.

In [ ]:
dashboard=make_subplots(rows=2,cols=2,subplot_titles=['Test metrics at 0.50','Observed churn mix','Capacity sweep (validation)','Top-15% test contact result'])
for metric in ['accuracy','precision','recall','f1','roc_auc']:
    dashboard.add_trace(go.Bar(x=[metric],y=[dense_metrics[metric]],name=metric,showlegend=False),row=1,col=1)
dashboard.add_trace(go.Bar(x=churn_counts['Churn'],y=churn_counts.Customers,marker_color=['#117c83','#e76f51'],showlegend=False),row=1,col=2)
dashboard.add_trace(go.Scatter(x=policy_df.contact_fraction,y=policy_df.precision,name='Precision',mode='lines+markers'),row=2,col=1)
dashboard.add_trace(go.Scatter(x=policy_df.contact_fraction,y=policy_df.recall,name='Recall',mode='lines+markers'),row=2,col=1)
dashboard.add_trace(go.Bar(x=['Test precision','Test recall'],y=[top15_precision,top15_recall],marker_color=['#166c9b','#117c83'],showlegend=False),row=2,col=2)
dashboard.update_layout(height=780,title_text=f'Assignment 4.1 | {STUDENT_NAME} | Customer churn')
dashboard.write_html(str(CHARTS/'A41_INTERACTIVE_DASHBOARD.html'))
display(dashboard)

# PART 2 — ASSIGNMENT 4.2
## LAB 13 — Why an entity embedding differs from one-hot encoding

A one-hot category uses one indicator column per possible value. An **embedding** is a short learnable vector looked up by category ID, similar to the word embeddings studied in Module 3. Here, each categorical feature receives its own embedding table. The continuous inputs still pass through the training-fitted numeric preprocessing pipeline.

If you are demonstrating Assignment 4.1 only, set `RUN_PART_2=False` in Lab 00 and run through the final export cells.

In [ ]:
if RUN_PART_2:
    categorical_encoder=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
    raw_cats_train=df_train[CATEGORICAL].fillna('__MISSING__').astype(str)
    categorical_encoder.fit(raw_cats_train)
    embedding_categories=[len(cats)+1 for cats in categorical_encoder.categories_]
    def categories_for(part):
        z=categorical_encoder.transform(part[CATEGORICAL].fillna('__MISSING__').astype(str)).astype('int32')+1
        return [z[:,i].reshape(-1,1) for i in range(z.shape[1])]
    num_imputer=SimpleImputer(strategy='median');num_scaler=StandardScaler()
    emb_num_train=num_scaler.fit_transform(num_imputer.fit_transform(df_train[NUMERIC])).astype('float32')
    def numeric_for(part):
        return num_scaler.transform(num_imputer.transform(part[NUMERIC])).astype('float32')
    def embedding_inputs(part):
        return [numeric_for(part)]+categories_for(part)
    emb_train=embedding_inputs(df_train);emb_val=embedding_inputs(df_val);emb_test=embedding_inputs(df_test)
    import joblib
    joblib.dump({'ordinal_encoder':categorical_encoder,'num_imputer':num_imputer,'num_scaler':num_scaler,
                 'numeric':NUMERIC,'categorical':CATEGORICAL},MODELS/'A42_embedding_preprocessing.joblib')
    print('Embedding table sizes:',dict(zip(CATEGORICAL,embedding_categories)))
else:
    print('Part 2 disabled. The final package will contain Assignment 4.1 evidence only.')

## LAB 14 — Train the embedding network

Each categorical column uses a small Keras `Embedding` layer and `Flatten`. All learned vectors join the scaled numerical data before the dense layers. These settings are classroom examples, not a claim that embeddings always outperform one-hot encoding.

In [ ]:
if RUN_PART_2:
    tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(RANDOM_SEED)
    numeric_input=tf.keras.Input(shape=(len(NUMERIC),),dtype='float32',name='numeric_values')
    all_inputs=[numeric_input];learned=[numeric_input]
    for idx,(field,size) in enumerate(zip(CATEGORICAL,embedding_categories)):
        inp=tf.keras.Input(shape=(1,),dtype='int32',name=f'category_{idx}')
        dim=min(4,max(2,int(math.ceil(math.sqrt(size)))))
        e=tf.keras.layers.Embedding(input_dim=size,output_dim=dim,name=f'embedding_{idx}')(inp)
        e=tf.keras.layers.Flatten()(e);all_inputs.append(inp);learned.append(e)
    joined=tf.keras.layers.Concatenate()(learned)
    h=tf.keras.layers.Dense(64,activation='relu')(joined)
    h=tf.keras.layers.Dropout(.25)(h)
    h=tf.keras.layers.Dense(32,activation='relu')(h)
    final=tf.keras.layers.Dense(1,activation='sigmoid')(h)
    embedding_net=tf.keras.Model(all_inputs,final,name='A42_entity_embedding_churn')
    embedding_net.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='binary_crossentropy',metrics=['accuracy'])
    embedding_net.summary()
    t0=time.perf_counter()
    emb_history=embedding_net.fit(emb_train,y_train,validation_data=(emb_val,y_val),
        epochs=EPOCHS,batch_size=BATCH_SIZE,verbose=1,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=PATIENCE,restore_best_weights=True)])
    embedding_seconds=time.perf_counter()-t0
    embedding_net.save(MODELS/'A42_entity_embedding_network.keras')
    pd.DataFrame(emb_history.history).to_csv(TABLES/'A42_embedding_training_history.csv',index=False)
    embedding_prob=embedding_net.predict(emb_test,verbose=0).reshape(-1)
    embedding_metrics=scores_at_cutoff(y_test,embedding_prob,.5)
    print('Embedding test metrics:',embedding_metrics)

## LAB 15 — Train the gradient-boosted tree baseline

A tree learns branching rules such as “tenure below six months.” Gradient boosting adds many trees that improve an ensemble's errors. The experiment uses **the exact same original feature columns and the same customer rows** as the neural networks. The boosting model receives the already fitted one-hot feature matrix for a controlled feature-information comparison.

In [ ]:
if RUN_PART_2:
    boosting=HistGradientBoostingClassifier(max_iter=(45 if DEMO_MODE else 180),learning_rate=.06,
        max_leaf_nodes=15,l2_regularization=.10,early_stopping=True,random_state=RANDOM_SEED)
    t0=time.perf_counter();boosting.fit(X_train,y_train)
    boosting_seconds=time.perf_counter()-t0
    boosting_prob=boosting.predict_proba(X_test)[:,1]
    boosting_metrics=scores_at_cutoff(y_test,boosting_prob,.5)
    import joblib
    joblib.dump(boosting,MODELS/'A42_gradient_boosting.joblib')
    print('Boosting test metrics:',boosting_metrics)
    print('Actual boosting iterations / trees:',boosting.n_iter_)

## LAB 16 — All models, same held-out test customers

Assignment 4.2 requires **all five metrics**, train time, and model complexity. Model size here is deliberately reported in different units: neural networks have trainable numeric parameters; gradient boosting contains trees. These counts should not be compared as if one parameter equals one tree.

In [ ]:
if RUN_PART_2:
    comparison_rows=[]
    for model,met,seconds,size,size_unit in [
        ('A41 Dense one-hot',dense_metrics,dense_seconds,dense_net.count_params(),'parameters'),
        ('A42 Entity embeddings',embedding_metrics,embedding_seconds,embedding_net.count_params(),'parameters'),
        ('A42 Gradient boosting',boosting_metrics,boosting_seconds,boosting.n_iter_,'trees')]:
        comparison_rows.append({'model':model,**met,'train_seconds':seconds,'model_size':int(size),'size_unit':size_unit})
    comparison=pd.DataFrame(comparison_rows)
    comparison.to_csv(TABLES/'REQUIRED_A42_three_model_comparison.csv',index=False)
    display(comparison.round(3))
    fig=px.bar(comparison,x='model',y=['accuracy','precision','recall','f1','roc_auc'],barmode='group',
      title='Assignment 4.2: five test metrics, shared test population')
    fig.write_html(str(CHARTS/'A42_INTERACTIVE_MODEL_COMPARISON.html'))
    display(fig)
    fig2=px.bar(comparison,x='model',y='train_seconds',title='Training wall time in this Colab runtime')
    fig2.write_html(str(CHARTS/'A42_INTERACTIVE_TRAINING_TIME.html'))
    display(fig2)

## LAB 17 — Confusion matrices and ROC curves for all models

The confusion matrix answers which mistakes are made at the chosen **0.50** cutoff. The ROC curve shows ranking across many thresholds. Because the business has limited contact capacity, an ROC-AUC win alone does not determine whom to deploy.

In [ ]:
if RUN_PART_2:
    model_probs={'Dense one-hot':dense_prob,'Entity embeddings':embedding_prob,'Gradient boosting':boosting_prob}
    fig,axs=plt.subplots(1,3,figsize=(15,4.5))
    for ax,(name,prob) in zip(axs,model_probs.items()):
        c=confusion_matrix(y_test,prob>=.5,labels=[0,1])
        ConfusionMatrixDisplay(c,display_labels=['Stayed','Churned']).plot(ax=ax,cmap='Blues',colorbar=False,values_format='d')
        ax.set_title(name)
    fig.tight_layout();fig.savefig(CHARTS/'A42_three_confusion_matrices.png',dpi=160);plt.show()
    plt.figure(figsize=(7,5))
    for name,prob in model_probs.items():
        fpr,tpr,_=roc_curve(y_test,prob)
        plt.plot(fpr,tpr,label=f'{name} (AUC={roc_auc_score(y_test,prob):.3f})')
    plt.plot([0,1],[0,1],'k--',alpha=.4);plt.xlabel('False-positive rate');plt.ylabel('True-positive rate')
    plt.title('Held-out test ROC curves');plt.legend();plt.tight_layout()
    plt.savefig(CHARTS/'A42_ROC_curves.png',dpi=160);plt.show()

## LAB 18 — Interactive model dashboard and business ranking comparison

The dashboard can be opened later as an HTML file. It lets students visually compare the five measured metrics and see how frequently each model would select the same customers when constrained to a top-15% contact budget.

In [ ]:
if RUN_PART_2:
    selections={name:rank_policy(y_test,prob)[0] for name,prob in model_probs.items()}
    select_tbl=pd.DataFrame({'customerID':df_test.customerID,'actual_churn':y_test})
    for name,flags in selections.items():
        select_tbl['selected_'+re.sub(r'[^a-z]+','_',name.lower()).strip('_')]=flags
    select_tbl.to_csv(TABLES/'A42_model_contact_selection_overlap.csv',index=False)
    rows=[]
    for name,flags in selections.items():
        rows.append({'model':name,'test_contacts':int(flags.sum()),
                     'precision_at_15pct':precision_score(y_test,flags,zero_division=0),
                     'recall_at_15pct':recall_score(y_test,flags,zero_division=0)})
    contact_comparison=pd.DataFrame(rows)
    contact_comparison.to_csv(TABLES/'A42_top15_contact_comparison.csv',index=False)
    display(contact_comparison.round(3))
    combo=make_subplots(rows=1,cols=2,subplot_titles=['Same held-out test metrics','Same 15% contact budget'])
    for m in ['precision','recall','f1','roc_auc']:
        combo.add_trace(go.Bar(x=comparison.model,y=comparison[m],name=m),row=1,col=1)
    for m in ['precision_at_15pct','recall_at_15pct']:
        combo.add_trace(go.Bar(x=contact_comparison.model,y=contact_comparison[m],name=m),row=1,col=2)
    combo.update_layout(barmode='group',height=570,title_text=f'Assignment 4.2 executive dashboard | {STUDENT_NAME}')
    combo.write_html(str(CHARTS/'A42_EXECUTIVE_DASHBOARD.html'))
    display(combo)

## LAB 19 — Interactive prediction sandbox (optional classroom exercise)

Change values in the controls to study how the original dense model scores **one fictional customer profile**. All fields are supplied by a complete example row from the IBM test partition; we modify only visible fields, then transform with the saved training pipeline. Do not treat a single predicted score as proof of causation or an offer's effectiveness.

In [ ]:
try:
    import ipywidgets as widgets
    base_profile=df_test[FEATURES].iloc[[0]].copy()
    tenure_widget=widgets.IntSlider(value=int(base_profile.tenure.iloc[0]),min=0,max=72,description='Tenure')
    charge_widget=widgets.FloatSlider(value=float(base_profile.MonthlyCharges.iloc[0]),min=15,max=125,step=2.5,description='Monthly $')
    contract_options=sorted(df.Contract.dropna().unique().tolist())
    contract_widget=widgets.Dropdown(options=contract_options,value=base_profile.Contract.iloc[0],description='Contract')
    run_button=widgets.Button(description='Predict churn score',button_style='info')
    message=widgets.Output()
    def run_case(_):
        with message:
            message.clear_output()
            current=base_profile.copy()
            current.loc[current.index[0],'tenure']=tenure_widget.value
            current.loc[current.index[0],'MonthlyCharges']=charge_widget.value
            current.loc[current.index[0],'Contract']=contract_widget.value
            p=float(dense_net.predict(prep.transform(current).astype('float32'),verbose=0).reshape(-1)[0])
            print(f'Illustrative customer score: {p:.3f}; this is an estimate, not a guarantee.')
    run_button.on_click(run_case)
    display(widgets.VBox([tenure_widget,charge_widget,contract_widget,run_button,message]))
except Exception as e:
    print('Optional widgets unavailable:',type(e).__name__,str(e)[:200])

## LAB 20 — Save run information and self-check evidence

The proof files summarize the experiment settings and verify that the splits, predictions and required output tables were generated. Full mode must be used for a final graded run. Demo mode is clearly labeled in every report and the ZIP README.

In [ ]:
run_manifest={'student':STUDENT_NAME,'course':COURSE,'module':MODULE,'company':COMPANY,
  'assignment_parts':['4.1']+(['4.2'] if RUN_PART_2 else []),'instructor_demo_mode':bool(DEMO_MODE),
  'source_dataset':'IBM Telco Customer Churn sample','dataset_sha256':DATA_SHA256,
  'dataset_rows':len(df),'seed':RANDOM_SEED,'contact_fraction':CONTACT_FRACTION,
  'train_rows':len(train_idx),'validation_rows':len(val_idx),'test_rows':len(test_idx),
  'tensorflow':tf.__version__,'timestamp_utc':datetime.now(timezone.utc).isoformat(),
  'a41_dense_test_metrics':dense_metrics,'a41_top15_policy':capacity_result}
if RUN_PART_2:
    run_manifest['a42_comparison']=comparison.to_dict(orient='records')
(PROOF/'REQUIRED_experiment_manifest.json').write_text(json.dumps(run_manifest,indent=2))
requirements={'A41_prediction_table':TABLES/'A41_test_prediction_evidence.csv',
              'A41_contact_policy':PROOF/'REQUIRED_A41_contact_policy.json',
              'A41_model':MODELS/'A41_dense_onehot.keras',
              'split_evidence':TABLES/'REQUIRED_reproducible_customer_splits.csv'}
if RUN_PART_2:
    requirements.update({'A42_comparison':TABLES/'REQUIRED_A42_three_model_comparison.csv',
                         'A42_embedding_model':MODELS/'A42_entity_embedding_network.keras',
                         'A42_boosted_model':MODELS/'A42_gradient_boosting.joblib'})
verification={k:{'exists':p.exists(),'bytes':p.stat().st_size if p.exists() else 0} for k,p in requirements.items()}
(PROOF/'REQUIRED_artifact_verification.json').write_text(json.dumps(verification,indent=2))
display(pd.DataFrame.from_dict(verification,orient='index'))
assert all(x['exists'] and x['bytes']>0 for x in verification.values()),'Some required evidence files are missing.'

## LAB 21 — Student-written analysis fields

**Important:** The notebook calculates the numbers automatically, but students must explain what they mean. Fill the text fields below using **your own reasoning and results**. The report will preserve blank fields as `[STUDENT RESPONSE REQUIRED]`; it will never invent the student's conclusions. For the original assignment, the 4.1 report should reach approximately **two pages of substantive writing**.

In [ ]:
A41_CLEANING_EXPLANATION = """"""  # Explain handling of TotalCharges, feature types, leakage prevention.
A41_CONFUSION_BUSINESS_COSTS = """"""  # Explain cost of TP, FP, FN, TN without invented dollar claims.
A41_THRESHOLD_JUSTIFICATION = """"""  # Explain precisely how the top 15% were selected.
A41_INTERPRETATION = """"""  # Two-page combined report: explain findings, weaknesses and next steps.

A42_EMBEDDINGS_INTERPRETATION = """"""  # Did embeddings improve on the original dense model?
A42_DEPLOYMENT_RECOMMENDATION = """"""  # Which model would you deploy, and why?
A42_CHANGE_MY_MIND_MEASUREMENT = """"""  # Name one future measurement that would reverse the choice.
A42_DEEP_LEARNING_JUDGMENT = """"""  # Direct paragraph: did deep learning earn its place?

analysis_fields={'A41 cleaning':A41_CLEANING_EXPLANATION,'A41 confusion business costs':A41_CONFUSION_BUSINESS_COSTS,
    'A41 15 percent policy':A41_THRESHOLD_JUSTIFICATION,'A41 findings':A41_INTERPRETATION}
if RUN_PART_2:
    analysis_fields.update({'A42 embeddings':A42_EMBEDDINGS_INTERPRETATION,'A42 deployment':A42_DEPLOYMENT_RECOMMENDATION,
                            'A42 decisive measurement':A42_CHANGE_MY_MIND_MEASUREMENT,
                            'A42 deep learning judgment':A42_DEEP_LEARNING_JUDGMENT})
missing_writing=[name for name,value in analysis_fields.items() if not str(value).strip()]
print('Missing written sections:',missing_writing if missing_writing else 'None')

## LAB 22 — Generate both named Word reports

This lab **creates the Word documents** automatically and imports the measured tables. Students fill any remaining placeholders in Word or, preferably, in LAB 21 before regenerating. Missing sections are clearly marked. A report with missing writing is a draft, **not a complete assignment submission**.

In [ ]:
def response(value):
    return str(value).strip() if str(value).strip() else '[STUDENT RESPONSE REQUIRED]'
def put_table(document,frame):
    t=document.add_table(rows=1,cols=len(frame.columns));t.style='Light Shading Accent 1'
    for j,col in enumerate(frame.columns):t.rows[0].cells[j].text=str(col)
    for _,row in frame.iterrows():
        cells=t.add_row().cells
        for j,val in enumerate(row):
            cells[j].text=f'{val:.3f}' if isinstance(val,(float,np.floating)) else str(val)

def heading_report(title):
    d=Document();d.add_heading(f'{COURSE} — {title}',0)
    d.add_paragraph(f'Student: {STUDENT_NAME} | Module: {MODULE} | Company case: {COMPANY}')
    d.add_paragraph(f'Experiment: {"SHORT INSTRUCTOR DEMO (not a final run)" if DEMO_MODE else "FULL EXPERIMENT"}; seed {RANDOM_SEED}; 7,043 customer IBM sample.')
    return d

doc=heading_report('Assignment 4.1: Churn and the Metric That Matters')
doc.add_heading('Dataset and preprocessing',level=1);doc.add_paragraph(response(A41_CLEANING_EXPLANATION))
doc.add_heading('Held-out test set metrics at a 0.50 threshold',level=1)
put_table(doc,pd.DataFrame([{'metric':k,'test_value':v} for k,v in dense_metrics.items()]))
doc.add_heading('Four outcomes and business costs',level=1)
doc.add_paragraph(f'At 0.50: true negatives {cm[0,0]}, false positives {cm[0,1]}, false negatives {cm[1,0]}, true positives {cm[1,1]}.')
doc.add_paragraph(response(A41_CONFUSION_BUSINESS_COSTS))
doc.add_heading('Top 15 percent customer contact decision',level=1)
doc.add_paragraph(f'The top-k policy contacted {test_k} of {len(y_test)} held-out customers. Precision: {top15_precision:.3f}; recall: {top15_recall:.3f}. The validation boundary score was {val_boundary:.3f}.')
doc.add_paragraph(response(A41_THRESHOLD_JUSTIFICATION))
doc.add_heading('Interpretation, limitations, and conclusion',level=1);doc.add_paragraph(response(A41_INTERPRETATION))
doc.add_paragraph('Supporting evidence: executed Colab notebook, split IDs, cleaning log, confusion plots, curves, and saved predictions in the submission package.')
path_a41=REPORTS/f'{SAFE_NAME}_MAI510_Assignment_4_1_Report.docx';doc.save(path_a41)

if RUN_PART_2:
    doc2=heading_report('Assignment 4.2: Embeddings Against Gradient Boosting')
    doc2.add_heading('Controlled comparison',level=1)
    doc2.add_paragraph('All models were evaluated on identical held-out customer IDs with identical feature information; classification metrics below use the same 0.50 score threshold.')
    put_table(doc2,comparison[['model','accuracy','precision','recall','f1','roc_auc','train_seconds','model_size','size_unit']].round(3))
    doc2.add_heading('Did entity embeddings help?',level=1);doc2.add_paragraph(response(A42_EMBEDDINGS_INTERPRETATION))
    doc2.add_heading('Which model should the company deploy?',level=1);doc2.add_paragraph(response(A42_DEPLOYMENT_RECOMMENDATION))
    doc2.add_heading('What evidence would change that decision?',level=1);doc2.add_paragraph(response(A42_CHANGE_MY_MIND_MEASUREMENT))
    doc2.add_heading('Did deep learning earn its place?',level=1);doc2.add_paragraph(response(A42_DEEP_LEARNING_JUDGMENT))
    doc2.add_paragraph('Supporting evidence: three-model comparison table, training times, charts, saved networks and tree model in the submission package.')
    path_a42=REPORTS/f'{SAFE_NAME}_MAI510_Assignment_4_2_Report.docx';doc2.save(path_a42)
print('Reports saved:',path_a41,('and '+str(path_a42)) if RUN_PART_2 else '')
if missing_writing:print('ACTION REQUIRED: Complete the marked writing in LAB 21 and rerun this cell.')

## LAB 23 — Optional: include your executed notebook in the final ZIP

**Google Colab does not expose a stable, supported way for every runtime to export its own executed notebook automatically.** To include that important academic artifact: choose **File → Download → Download .ipynb** from Colab, then run this optional cell and upload the downloaded `.ipynb`. The ZIP will contain it under `notebooks/`. If you skip this, the ZIP will say clearly that the executed notebook must be submitted separately.

In [ ]:
NOTEBOOK_DIR=ART/'notebooks';NOTEBOOK_DIR.mkdir(exist_ok=True)
UPLOAD_EXECUTED_NOTEBOOK = False  # Set True after downloading this notebook from Colab.
if UPLOAD_EXECUTED_NOTEBOOK:
    try:
        from google.colab import files
        newfiles=files.upload()
        for filename,content in newfiles.items():
            if filename.lower().endswith('.ipynb'):
                (NOTEBOOK_DIR/Path(filename).name).write_bytes(content)
                print('Saved executed notebook:',filename)
    except ImportError:print('Upload is designed for Google Colab.')
else:
    print('Notebook upload skipped. Download and submit executed .ipynb separately, or rerun this cell with True.')

# FINAL STEP — LAB 24: Complete submission ZIP and download

The last cell packages **all generated evidence**, including the uploaded source CSV, reproducible split assignments, trained models, fitted preprocessing, plots, interactive HTML dashboards, machine-readable proof and named Word reports.

It warns you about missing written answers or a missing executed notebook. The ZIP does **not** claim the assignment is complete when those requirements are outstanding. Students should supply the notebook and complete the written report before submission.

In [ ]:
readme=f"""MAI 510 — MODULE 4 SUBMISSION PACKAGE
Student: {STUDENT_NAME}
Experiment: {'SHORT INSTRUCTOR DEMO — NOT A FINAL GRADED RUN' if DEMO_MODE else 'FULL EXPERIMENT'}
Assignment parts: {'4.1 + 4.2' if RUN_PART_2 else '4.1 only'}
Dataset: IBM public Telco Customer Churn sample, 7,043 rows
Source: https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv
Split seed: {RANDOM_SEED}; test population: {len(y_test)}

Generated folders:
charts/ = saved PNG figures + interactive HTML dashboards
tables/ = cleaning log, split IDs, all measured metrics, score evidence
models/ = trained Keras networks, boosting model, fitted preprocessors
reports/ = named Word documents with data; student analysis must be completed
proof/ = run metadata and required evidence checks
notebooks/ = optional executed notebook uploaded by student

Missing narrative fields: {', '.join(missing_writing) if missing_writing else 'none'}
Executed notebook included: {bool(list(NOTEBOOK_DIR.glob('*.ipynb')))}

IMPORTANT: Review your own calculated figures and written analysis before submitting.
A missing notebook or narrative field means this ZIP is an incomplete submission.
"""
(ART/'README_SUBMISSION.txt').write_text(readme)
zip_name=f'{SAFE_NAME}_MAI510_Module4_{"A41_A42" if RUN_PART_2 else "A41"}_SUBMISSION.zip'
zip_path=ROOT/zip_name
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(ART.rglob('*')):
        if p.is_file():z.write(p,arcname=p.relative_to(ART))
with zipfile.ZipFile(zip_path) as z:
    names=z.namelist()
    assert any(x.endswith('README_SUBMISSION.txt') for x in names)
    assert any(x.endswith('REQUIRED_reproducible_customer_splits.csv') for x in names)
    assert any(x.endswith('.keras') for x in names)
    assert any(x.endswith('.docx') for x in names)
print('PACKAGE CREATED:',zip_path)
print('ZIP files:',len(names),'ZIP size (MB):',round(zip_path.stat().st_size/1e6,2))
print('Written fields incomplete:',len(missing_writing),'| Executed notebook attached:',bool(list(NOTEBOOK_DIR.glob('*.ipynb'))))
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Download path:',zip_path)